### Run Dependencies

In [ ]:
%run Legal_-_Bronze_To_Silver

### Load All Silver Tables

In [ ]:
# ── Load all Silver tables from managed Lakehouse tables ──────────────────────
#    No path strings needed — just schema.table_name

Silver_raw_billing_invoice      = spark.read.table(f"{Silver_Schema}.raw_billing_invoice")
Silver_raw_client_master        = spark.read.table(f"{Silver_Schema}.raw_client_master")
Silver_raw_client_response_log  = spark.read.table(f"{Silver_Schema}.raw_client_response_log")
Silver_raw_client_update_log    = spark.read.table(f"{Silver_Schema}.raw_client_update_log")
Silver_raw_document_access_log  = spark.read.table(f"{Silver_Schema}.raw_document_access_log")
Silver_raw_document_metadata    = spark.read.table(f"{Silver_Schema}.raw_document_metadata")
Silver_raw_document_search_log  = spark.read.table(f"{Silver_Schema}.raw_document_search_log")
Silver_raw_lawyer_availability  = spark.read.table(f"{Silver_Schema}.raw_lawyer_availability")
Silver_raw_lawyer_profile       = spark.read.table(f"{Silver_Schema}.raw_lawyer_profile")
Silver_raw_matter_registry      = spark.read.table(f"{Silver_Schema}.raw_matter_registry")
Silver_raw_meeting_action_item  = spark.read.table(f"{Silver_Schema}.raw_meeting_action_item")
Silver_raw_meeting_transcript   = spark.read.table(f"{Silver_Schema}.raw_meeting_transcript")
Silver_raw_resource_allocation  = spark.read.table(f"{Silver_Schema}.raw_resource_allocation")
Silver_raw_timesheet_entry      = spark.read.table(f"{Silver_Schema}.raw_timesheet_entry")
Silver_raw_unlogged_work_signal = spark.read.table(f"{Silver_Schema}.raw_unlogged_work_signal")

print("All Silver tables loaded.")


### Gold Table Registry

In [ ]:
# ── Gold table registry ────────────────────────────────────────────────────────
#    Each entry: (gold_table_name, silver_df, source_file_name, table_type, cols_to_drop)
#    cols_to_drop: columns added in Bronze/Silver that should not appear in Gold

GOLD_TABLE_REGISTRY = [
    # ── DIMENSIONS ────────────────────────────────────────────────────────────
    ("Lawyer_Master_Data",              Silver_raw_lawyer_profile,       "raw_lawyer_profile.csv",         "DIM", ["Comments"]),
    ("Client_Master_Data",              Silver_raw_client_master,         "raw_client_master.csv",           "DIM", ["Comments"]),
    ("Matter_Master_Data",              Silver_raw_matter_registry,       "raw_matter_registry.csv",         "DIM", ["Comments"]),
    ("Document_Master_Data",            Silver_raw_document_metadata,     "raw_document_metadata.csv",       "DIM", ["Comments"]),
    ("Meeting_Master_Data",             Silver_raw_meeting_transcript,    "raw_meeting_transcript.csv",      "DIM", ["Comments"]),
    ("Billing_Master_Data",             Silver_raw_billing_invoice,       "raw_billing_invoice.csv",         "DIM", ["Comments"]),
    ("Lawyer_Availability_Master_Data", Silver_raw_lawyer_availability,   "raw_lawyer_availability.csv",     "DIM", ["Comments"]),
    # ── FACTS ─────────────────────────────────────────────────────────────────
    ("Fact_Utilisation",                Silver_raw_timesheet_entry,       "raw_timesheet_entry.csv",         "FACT", ["Comments", "Uploaded_Datetime"]),
    ("Fact_Team_Allocation",            Silver_raw_resource_allocation,   "raw_resource_allocation.csv",     "FACT", ["Comments", "Uploaded_Datetime"]),
    ("Fact_Search_Analytics",           Silver_raw_document_search_log,   "raw_document_search_log.csv",     "FACT", ["Comments", "Uploaded_Datetime"]),
    ("Fact_Meeting_Automation",         Silver_raw_meeting_transcript,    "raw_meeting_transcript.csv",      "FACT", ["Comments", "Uploaded_Datetime"]),
    ("Fact_Client_Update",              Silver_raw_client_update_log,     "raw_client_update_log.csv",       "FACT", ["Comments", "Uploaded_Datetime"]),
    ("Fact_Client_Response",            Silver_raw_client_response_log,   "raw_client_response_log.csv",     "FACT", ["Comments", "Uploaded_Datetime"]),
    ("Fact_Document_Access",            Silver_raw_document_access_log,   "raw_document_access_log.csv",     "FACT", ["Comments", "Uploaded_Datetime"]),
    ("Fact_Meeting_Action_Item",        Silver_raw_meeting_action_item,   "raw_meeting_action_item.csv",     "FACT", ["Comments", "Uploaded_Datetime"]),
    ("Fact_Unlogged_Work",              Silver_raw_unlogged_work_signal,  "raw_unlogged_work_signal.csv",    "FACT", ["Comments", "Uploaded_Datetime"]),
]

print(f"Gold registry loaded | {len(GOLD_TABLE_REGISTRY)} tables")


### `promote_to_gold()` Helper Function

In [ ]:
import uuid

def promote_to_gold(gold_table_name, silver_df, source_file_name, table_type, cols_to_drop):
    """
    Reads a Silver DataFrame, drops housekeeping columns,
    writes a managed Gold Lakehouse table, and updates audit + metadata logs.

    Parameters
    ----------
    gold_table_name  : str   e.g. "Lawyer_Master_Data"
    silver_df        : DataFrame already loaded from Silver
    source_file_name : str   e.g. "raw_lawyer_profile.csv"
    table_type       : str   "DIM" or "FACT"
    cols_to_drop     : list  columns to remove before writing to Gold

    Returns
    -------
    int  — record count written (0 on failure)
    """
    _audit_id   = str(uuid.uuid4())
    _gold_table = f"{Gold_Schema}.{gold_table_name}"

    try:
        # Drop housekeeping columns (ignore missing ones silently)
        _existing = silver_df.columns
        _drop     = [c for c in cols_to_drop if c in _existing]
        df        = silver_df.drop(*_drop)
        df.createOrReplaceTempView(gold_table_name)

        df.cache()
        record_count = df.count()

        # Write managed Gold table
        df.write \
            .mode("overwrite") \
            .format("delta") \
            .option("mergeSchema", "true") \
            .saveAsTable(_gold_table)

        df.unpersist()
        print(f"[GOLD] Written | table={_gold_table} | records={record_count} | type={table_type}")

        # Audit log
        try:
            log_audit(
                audit_id          = _audit_id,
                source_type       = "Lakehouse",
                destination       = _gold_table,
                notebook_name     = "Legal - Silver_To_Gold",
                layer_name        = "GOLD",
                table_name        = gold_table_name,
                records_processed = record_count,
                status            = "SUCCESS",
                error_message     = "",
            )
        except Exception as _ae:
            print(f"[AUDIT WARN] {gold_table_name} | {_ae}")

        # Metadata log
        update_layer_metadata(
            layer      = "GOLD",
            file_name  = source_file_name,
            table_name = gold_table_name,
            schema     = Gold_Schema,
            table_type = table_type,
        )

        return record_count

    except Exception as _e:
        try:
            log_audit(
                audit_id          = _audit_id,
                source_type       = "Lakehouse",
                destination       = _gold_table,
                notebook_name     = "Legal - Silver_To_Gold",
                layer_name        = "GOLD",
                table_name        = gold_table_name,
                records_processed = 0,
                status            = "FAILED",
                error_message     = str(_e),
            )
        except Exception as _ae2:
            print(f"[AUDIT WARN] {_ae2}")
        print(f"[FAILED] {gold_table_name} | {_e}")
        return 0


print("promote_to_gold() registered.")


### Promote All Tables — Silver → Gold

In [ ]:
# ── Promote all Gold tables ────────────────────────────────────────────────────
gold_record_counts = {}

for _entry in GOLD_TABLE_REGISTRY:
    _gold_name, _silver_df, _src_file, _tbl_type, _drop_cols = _entry
    gold_record_counts[_gold_name] = promote_to_gold(
        gold_table_name  = _gold_name,
        silver_df        = _silver_df,
        source_file_name = _src_file,
        table_type       = _tbl_type,
        cols_to_drop     = _drop_cols,
    )

total_gold_records = sum(gold_record_counts.values())
print(f"\n[GOLD SUMMARY] Total records across all Gold tables: {total_gold_records}")
for _t, _c in gold_record_counts.items():
    print(f"  {_t:45s} {_c:>8,}")


### Date Dimension

In [ ]:
# ── Date Dimension ─────────────────────────────────────────────────────────────
_audit_id  = str(uuid.uuid4())
_gold_name = "Date_Dimension_Master_Data"
_gold_table = f"{Gold_Schema}.{_gold_name}"

try:
    _start_year = datetime.now().year - 3
    _end_year   = datetime.now().year + 7

    Date_Dimension_Master_Data = (
        spark.sql(f"""
            SELECT explode(sequence(
                to_date('{_start_year}-01-01'),
                to_date('{_end_year}-12-31'),
                interval 1 day
            )) AS Date
        """)
        .withColumn("Day",        date_format(col("Date"), "d").cast("int"))
        .withColumn("Day_Name",   date_format(col("Date"), "EEEE"))
        .withColumn("Month",      month(col("Date")))
        .withColumn("Month_Name", date_format(col("Date"), "MMMM"))
        .withColumn("Year",       year(col("Date")))
        .withColumn("Quarter",    quarter(col("Date")))
        .withColumn("Is_Weekend", when(dayofweek(col("Date")).isin(1, 7), 1).otherwise(0))
        .withColumn("Season",
            when(col("Month").isin([12, 1, 2]),  "Winter")
            .when(col("Month").isin([3, 4, 5]),  "Spring")
            .when(col("Month").isin([6, 7, 8]),  "Summer")
            .otherwise("Autumn")
        )
    )

    _dim_count = Date_Dimension_Master_Data.count()

    Date_Dimension_Master_Data.write \
        .mode("overwrite") \
        .format("delta") \
        .option("mergeSchema", "true") \
        .saveAsTable(_gold_table)

    gold_record_counts[_gold_name] = _dim_count
    print(f"[GOLD] Written | table={_gold_table} | records={_dim_count}")

    try:
        log_audit(
            audit_id=_audit_id, source_type="GENERATED",
            destination=_gold_table, notebook_name="Legal - Silver_To_Gold",
            layer_name="GOLD", table_name=_gold_name,
            records_processed=_dim_count, status="SUCCESS", error_message="",
        )
    except Exception as _ae:
        print(f"[AUDIT WARN] {_ae}")

except Exception as _e:
    log_audit(
        audit_id=_audit_id, source_type="GENERATED",
        destination=_gold_table, notebook_name="Legal - Silver_To_Gold",
        layer_name="GOLD", table_name=_gold_name,
        records_processed=0, status="FAILED", error_message=str(_e),
    )
    print(f"[FAILED] {_gold_name} | {_e}")


### KPI Tables

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  KPI COMPUTATIONS  (saved as managed Gold tables)
# ══════════════════════════════════════════════════════════════════════════════

# ── KPI 1: Search Success Rate ─────────────────────────────────────────────────
Search_Success_Rate = (
    spark.read.table(f"{Gold_Schema}.Fact_Search_Analytics")
    .groupBy()
    .agg(
        (sum(when(col("search_status") == "SUCCESS", 1).otherwise(0))
         / count("*") * 100).alias("Search_Success_Rate_Pct")
    )
)
Search_Success_Rate.write.mode("overwrite").format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{Gold_Schema}.KPI_Search_Success_Rate")

# ── KPI 2: Average Search Time ─────────────────────────────────────────────────
Average_Search_Time = (
    spark.read.table(f"{Gold_Schema}.Fact_Search_Analytics")
    .groupBy()
    .agg(avg("total_search_duration_sec").alias("Avg_Search_Duration_Sec"))
)
Average_Search_Time.write.mode("overwrite").format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{Gold_Schema}.KPI_Average_Search_Time")

# ── KPI 3: Most Accessed Documents ────────────────────────────────────────────
Most_Accessed_Documents = (
    spark.read.table(f"{Gold_Schema}.Fact_Search_Analytics")
    .groupBy("matter_id")
    .agg(count("*").alias("Access_Count"))
    .orderBy(col("Access_Count").desc())
)
Most_Accessed_Documents.write.mode("overwrite").format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{Gold_Schema}.KPI_Most_Accessed_Documents")

# ── KPI 4: Client Update Automation Rate ──────────────────────────────────────
Client_Update_Automation_Rate = (
    spark.read.table(f"{Gold_Schema}.Fact_Client_Update")
    .groupBy()
    .agg(
        (sum(when(col("draft_source") == "AI", 1).otherwise(0))
         / count("*") * 100).alias("Automation_Rate_Pct")
    )
)
Client_Update_Automation_Rate.write.mode("overwrite").format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{Gold_Schema}.KPI_Client_Update_Automation_Rate")

# ── KPI 5: Billable Utilisation % ─────────────────────────────────────────────
Billable_Utilisation = (
    spark.read.table(f"{Gold_Schema}.Fact_Utilisation")
    .groupBy()
    .agg(
        (sum(when(col("billing_type") == "BILLABLE", col("hours_logged")).otherwise(0))
         / sum("hours_logged") * 100).alias("Billable_Utilisation_Pct")
    )
)
Billable_Utilisation.write.mode("overwrite").format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{Gold_Schema}.KPI_Billable_Utilisation")

# ── KPI 6: Revenue Per Legal Resource ─────────────────────────────────────────
Revenue_Per_Legal_Resource = (
    spark.read.table(f"{Gold_Schema}.Fact_Utilisation")
    .withColumn("Revenue", col("hours_logged") * col("billable_rate_usd"))
    .groupBy("user_id")
    .agg(Fun.sum("Revenue").alias("Total_Revenue_USD"))
)
Revenue_Per_Legal_Resource.write.mode("overwrite").format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{Gold_Schema}.KPI_Revenue_Per_Legal_Resource")

# ── KPI 7: Team Allocation Efficiency ─────────────────────────────────────────
Team_Allocation_Efficiency = (
    spark.read.table(f"{Gold_Schema}.Fact_Team_Allocation")
    .groupBy()
    .agg(avg("expertise_match_score").alias("Avg_Match_Score"))
)
Team_Allocation_Efficiency.write.mode("overwrite").format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{Gold_Schema}.KPI_Team_Allocation_Efficiency")

print("[KPIs] All 7 KPI tables written successfully.")


### Final Audit Log

In [ ]:
# ── Final summary audit log ────────────────────────────────────────────────────
try:
    log_audit(
        audit_id          = str(uuid.uuid4()),
        source_type       = "Lakehouse",
        destination       = f"{Gold_Schema}.*",
        notebook_name     = "Legal - Silver_To_Gold",
        layer_name        = "GOLD",
        table_name        = "ALL_GOLD_TABLES",
        records_processed = sum(gold_record_counts.values()),
        status            = "SUCCESS",
        error_message     = "",
    )
    print("[AUDIT] Final Gold summary log written.")
except Exception as _ae:
    print(f"[AUDIT WARN] Final summary: {_ae}")


### Optional Files Backup

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  OPTIONAL FILES BACKUP  —  Silver → Gold
# ══════════════════════════════════════════════════════════════════════════════
if ENABLE_FILES_BACKUP:
    print("[BACKUP] Starting Gold backup...")

    _gold_tables = [
        (name, Gold_Schema)
        for name, _, _, _, _ in GOLD_TABLE_REGISTRY
    ] + [("Date_Dimension_Master_Data", Gold_Schema)]

    backup_tables_to_files(
        tables      = _gold_tables,
        backup_root = Gold_Backup_Path,
        layer_label = "GOLD_BACKUP",
    )

    _prune_old_backups(Gold_Backup_Path, BACKUP_RETENTION_DAYS)

else:
    print("[BACKUP] Skipped — ENABLE_FILES_BACKUP is False")
